# Where do the errors come from?

Two suspects:

1. **the reader** (pdfplumber) — did it damage the document?
2. **the models** — did they label it wrong?

The same three-step check is run on two documents — and they reach **opposite
verdicts**.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    containment, extract_gold, resolve_old_gt_path, tokenize,
)

EXTRACTOR = 'pdfplumber'
MODELS = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']
pd.set_option('display.max_colwidth', 80)


def load(n):
    """Annotation, reader blocks, and each block's true label for sample n.

    A block's true label is the label of the annotation item its text belongs
    to. A block that does not fit any single item gets None — that only
    happens when one block spans two items at once.
    """
    gold = extract_gold(resolve_old_gt_path(n))
    blocks = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'sample{n}.json')
                        .read_text(encoding='utf-8'))
    truth = []
    for b in blocks:
        bt = tokenize(b['text'])
        # A one- or two-word block (a page number, a stray '1') can match an
        # item by accident, so tiny blocks are not given a true label at all.
        if len(bt) < 3:
            truth.append(None)
            continue
        best, lab, bi = 0.0, None, None
        for gi, (gt, gl) in enumerate(gold):
            c = containment(bt, tokenize(gt))
            if c > best:
                best, lab, bi = c, gl, gi
        if best < 0.75:
            truth.append(None)
            continue
        # A block that also swallows a DIFFERENT whole item (a heading fused
        # onto the front of an answer) spans two items — no single label can
        # be right, so it gets no true label either.
        spans_two = any(gi != bi and len(tokenize(gt)) >= 2
                        and containment(tokenize(gt), bt) >= 0.9
                        for gi, (gt, gl) in enumerate(gold))
        truth.append(None if spans_two else lab)
    return gold, blocks, truth


def load_labels(model, n):
    """The same blocks after one model labeled them."""
    return json.loads(P.labeled_path(P.make_tag(model, EXTRACTOR), n)
                      .read_text(encoding='utf-8'))


## Sample 1


### Step 1 — Did the reader damage sample 1?

Every block the reader produced is checked against the annotation: does its
text belong to **one** real item?


In [2]:
gold, blocks, truth = load(1)

lost = [i for i, t in enumerate(truth) if t is None]
print(f'the annotation has {len(gold)} items')
print(f'the reader produced {len(blocks)} blocks')
print(f'blocks left out of the check (too small to judge, or spanning',
      f'two items): {len(lost)}')
for i in lost:
    print(f'   block {i:>2}: {blocks[i]["text"][:66]!r}')


the annotation has 28 items
the reader produced 78 blocks
blocks left out of the check (too small to judge, or spanning two items): 2
   block  7: 'study.'
   block 21: 'Postures”.'


### Step 2 — How well did each model label the same blocks?


In [3]:
labeled = {m: load_labels(m, 1) for m in MODELS}
known = sum(1 for t in truth if t)
acc = pd.DataFrame([
    {'model': m,
     'correct': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t),
     'wrong': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') != t),
     'accuracy': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t) / known}
    for m in MODELS]).set_index('model')
display(acc.style.background_gradient(cmap='Greens', subset=['accuracy'])
        .format({'accuracy': '{:.0%}'}))


,correct,wrong,accuracy
model,,,
llama3.1:8b,65,11,86%
gemma4:e4b,76,0,100%
llama3.3:70b,75,1,99%


### Step 3 — The wrong blocks, one by one


In [4]:
for m in MODELS:
    bad = [{'true label': t, 'model said': b.get('label'), 'text': b['text']}
           for t, b in zip(truth, labeled[m]) if t and b.get('label') != t]
    print(f'{m}: {len(bad)} wrong')
    if bad:
        display(pd.DataFrame(bad))
    print()


llama3.1:8b: 11 wrong


,true label,model said,text
0,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for ..."
1,answer.text,section.description,about other studies.
2,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
3,answer.text,question.text,The following data will be created as a result of this project:
4,question.text,section.title,A. Repository where scientific data and metadata will be archived:
5,question.text,section.title,B. How scientific data will be findable and identifiable:
6,question.text,section.title,C. When and how long the scientific data will be made available:
7,question.text,section.title,"A. Factors affecting subsequent access, distribution, or reuse of scientific..."
8,answer.text,question.text,about using their data:
9,question.text,section.title,B. Whether access to scientific data will be controlled:



gemma4:e4b: 0 wrong

llama3.3:70b: 1 wrong


,true label,model said,text
0,answer.text,section.description,The following data will be created as a result of this project:


### Conclusion for sample 1

**The errors come from the model, not from pdfplumber.**

gemma4 received exactly the same blocks and labeled every one correctly — so the
input was good enough to score 100%, and every error here belongs to the model.

llama3.1's mistakes are mostly one repeated error: lettered lines like
`B. Scientific data that will be preserved…` are questions, but they *look* like
headings, and it calls them headings.


## Sample 6 — the opposite case

Same check, different document. This one's section headings are **underlined**
instead of bold.


### Step 1 — Did the reader damage sample 6?

Every block the reader produced is checked against the annotation: does its
text belong to **one** real item?


In [5]:
gold, blocks, truth = load(6)

lost = [i for i, t in enumerate(truth) if t is None]
print(f'the annotation has {len(gold)} items')
print(f'the reader produced {len(blocks)} blocks')
print(f'blocks left out of the check (too small to judge, or spanning',
      f'two items): {len(lost)}')
for i in lost:
    print(f'   block {i:>2}: {blocks[i]["text"][:66]!r}')


the annotation has 11 items
the reader produced 24 blocks
blocks left out of the check (too small to judge, or spanning two items): 8
   block  1: '1. Types of data. The bulk of the data generated in this project w'
   block  6: '2. Data and metadata standards. The PI’s research group will adopt'
   block 10: '3. Policies for access and sharing. Interested parties will be abl'
   block 12: 'execution.'
   block 13: '4. Policies and provisions for re-use, re-distribution. Simulation'
   block 18: '5. Plans for archiving and preservation of access. Local data will'
   block 22: '> 10 years.'
   block 23: '1'


### Step 2 — How well did each model label the same blocks?


In [6]:
labeled = {m: load_labels(m, 6) for m in MODELS}
known = sum(1 for t in truth if t)
acc = pd.DataFrame([
    {'model': m,
     'correct': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t),
     'wrong': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') != t),
     'accuracy': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t) / known}
    for m in MODELS]).set_index('model')
display(acc.style.background_gradient(cmap='Greens', subset=['accuracy'])
        .format({'accuracy': '{:.0%}'}))


,correct,wrong,accuracy
model,,,
llama3.1:8b,7,9,44%
gemma4:e4b,13,3,81%
llama3.3:70b,16,0,100%


### Step 3 — The wrong blocks, one by one


In [7]:
for m in MODELS:
    bad = [{'true label': t, 'model said': b.get('label'), 'text': b['text']}
           for t, b in zip(truth, labeled[m]) if t and b.get('label') != t]
    print(f'{m}: {len(bad)} wrong')
    if bad:
        display(pd.DataFrame(bad))
    print()


llama3.1:8b: 9 wrong


,true label,model said,text
0,answer.text,question.text,dimensional arrays of numbers and bytes representing fluid state variables g...
1,answer.text,question.text,"data using both commercial packages, such as Matlab and free visualization p..."
2,answer.text,section.description,practice that all generated data should be straightforwardly reproducible. T...
3,answer.text,question.text,stored within the same directory on a mass storage system along with the dat...
4,answer.text,question.text,the end of the project. Such parties will be responsible for their own versi...
5,answer.text,question.text,several terrabytes with an associated cost of storage. Data will be made pub...
6,answer.text,question.text,intellectual rights to the code will be retained by the CU group. Disseminat...
7,answer.text,question.text,"mass storage system. As mentioned above, all data generated is archived with..."
8,answer.text,question.text,NSF-funded research computing. We imagine the data being of relevance and st...



gemma4:e4b: 3 wrong


,true label,model said,text
0,answer.text,question.text,the end of the project. Such parties will be responsible for their own versi...
1,answer.text,question.text,several terrabytes with an associated cost of storage. Data will be made pub...
2,answer.text,question.text,"mass storage system. As mentioned above, all data generated is archived with..."



llama3.3:70b: 0 wrong



### Step 4 — What did the models do with the fused blocks?

The blocks left out above are the interesting ones: each holds an underlined
heading glued to the start of its answer. No single label can be right for them —
but what did the models actually say?


In [8]:
fused = [i for i, t in enumerate(truth) if t is None
         and len(blocks[i]['text'].split()) > 4]
rows = [{**{m: labeled[m][i].get('label') for m in MODELS},
         'text': blocks[i]['text']} for i in fused]
display(pd.DataFrame(rows, index=[f'block {i}' for i in fused]))


,llama3.1:8b,gemma4:e4b,llama3.3:70b,text
block 1,section.title,section.title,section.title,"1. Types of data. The bulk of the data generated in this project will be 1, ..."
block 6,section.title,section.title,section.title,2. Data and metadata standards. The PI’s research group will adopt the longs...
block 10,section.title,section.title,section.title,3. Policies for access and sharing. Interested parties will be able to reque...
block 13,section.title,section.title,section.title,"4. Policies and provisions for re-use, re-distribution. Simulation data will..."
block 18,section.title,section.title,section.title,5. Plans for archiving and preservation of access. Local data will be archiv...


**Every model says `section.title`, unanimously.** That is the sensible answer —
each block really does start with a heading. But the block also contains the
answer's first line, so the scoring can match it to neither item: the heading item
is missed *and* the answer loses its opening. The models did the best possible
thing with the input, and still lose twice.


### Step 5 — What those blocks become in the final output

The label applies to the **whole block**, so each model's produced "heading" is the
fused text — heading *plus* the answer's first line. The scorer accepts a match when
at least 75% of a produced item's words are inside the real item. Mostly answer
words, so:


In [9]:
gold_heads = [t for t, l in gold if l == 'section.title']


def produced_heads(model):
    s3 = json.loads(P.structured_path(P.make_tag(model, EXTRACTOR), 6)
                    .read_text(encoding='utf-8'))
    return [s.get('title', '') for s in s3['narrative']['template']['section']
            if s.get('title')]


for m in MODELS:
    heads = produced_heads(m)
    matched = sum(max((containment(tokenize(p), tokenize(g)) for g in gold_heads),
                      default=0) >= 0.75 for p in heads)
    print(f'{m:<14} real headings matched: {matched} of {len(gold_heads)}')

print()
rows = []
for p in produced_heads(MODELS[0]):
    if len(tokenize(p)) < 3:
        continue                      # stray page number, not a fused heading
    ov, g = max(((containment(tokenize(p), tokenize(g)), g) for g in gold_heads),
                key=lambda x: x[0], default=(0, ''))
    rows.append({'produced heading (fused)': p[:56],
                 'real heading': g, 'word overlap': f'{ov:.0%}',
                 'match (needs 75%)': 'no'})
display(pd.DataFrame(rows))


llama3.1:8b    real headings matched: 0 of 5
gemma4:e4b     real headings matched: 0 of 5
llama3.3:70b   real headings matched: 0 of 5



,produced heading (fused),real heading,word overlap,match (needs 75%)
0,1. Types of data. The bulk of the data generated in this,Types of data,19%,no
1,2. Data and metadata standards. The PI’s research group,Data and metadata standards,31%,no
2,3. Policies for access and sharing. Interested parties w,Policies for access and sharing,31%,no
3,"4. Policies and provisions for re-use, re-distribution.","Policies and provisions for re-use, re-distribution",47%,no
4,5. Plans for archiving and preservation of access. Local,Plans for archiving and preservation of access,41%,no


**All three models produce these same fused items, at the same overlap figures —
and match 0 of the 5 real headings.** An 8B and a 70B model ending up with the
identical wrong item is only possible when the item was decided by the input, not
by the model.


### Conclusion for sample 6

**Here the reader is at fault — and no model can fix it.**

This document's headings are underlined, and an underline is a drawn line the reader
cannot see. So each heading arrives **glued to its answer in one block** — two items,
one block, one label. Whichever label the model picks, the other item is lost.

The cleanest proof is llama3.3:70b: it labels **every judgeable block correctly**
(100% in step 2) and answers `section.title` on all five fused blocks — the sensible
call — yet this is still its worst document in the corpus (f1 0.52). A model with
zero labeling mistakes cannot score well here, because five of the eleven items
never reached it as separate blocks.

### The overall lesson

| document | who is at fault | how you can tell |
|---|---|---|
| sample 1 | the model | another model scored 100% from the same input |
| sample 6 | the reader | even a model with zero block mistakes scores only 0.52 |
